# 05a. Atmospheric and land initialization benefits for subseasonal 2-m temperature: daily curves (main figure)

## Purpose

How much skill does DART initialization of the atmosphere, and of the land, add to subseasonal 2-m temperature forecasts over northern mid-latitude land, as a function of lead week, under the prescribed observed SST of the F20TR hindcasts? A land-atmosphere analogue of the three-panel "initialization contribution" figure: (a) the hypothesis, (b) verification against ERA5, (c) verification against a held-out model member.

## Design

The experiments (`EXPERIMENTS` in `configs/fcst_init_experiment_config.py`) form two 2 x 2 designs of the atmosphere and land initial states, referred to by roles: the first digit is the atmosphere, the second the land (1 = DART). In both, E11 is DA_COUPLED (the DART analyses of a coupled land-atmosphere assimilation), and E10 and E01 each exchange one component of it with the reference E00, to understand the consistency between the land and atmosphere initial states. `CONFIG_GROUP` picks the design (`INIT_BENEFIT_DESIGNS` in `configs/hind_lead_eval_cases.py`). Each experiment is a 20-member ensemble whose members start from (each case's `user_nl_eam` ncdata and `user_nl_elm` finidat):

**`dart_vs_capt`** (default; question c, DART vs a CAPT-like initialization):

| Role | Configuration | Atmosphere | Land | Land-atmosphere states |
|---|---|---|---|---|
| E00 | ERA5CLIMFILL | ERA5 completed with CLIM_BASE fields (EN01) and the same + 19 DART perturbations (EN02-EN20) | DATM-forced land ensemble | consistent (reference) |
| E10 | DA_ATM | the 20 DART analyses of E11 | the DATM-forced land of E00 | mixed |
| E01 | DA_LND_CLIMFILL | the 20 atmospheres of E00 (in another order of the member labels) | the 20 DART land analyses of E11 | mixed |
| E11 | DA_COUPLED | 20 DART analyses | 20 DART land analyses | consistent (coupled DA) |

CLIM_BASE (no assimilation) is outside this 2 x 2.

**`lnd_vs_atm`** (question b, land vs atmosphere initialization; the component swaps of DA_COUPLED):

| Role | Configuration | Atmosphere | Land | Land-atmosphere states |
|---|---|---|---|---|
| E00 | CLIM_BASE | FREE_CLIM + 19 DART perturbations (EN02-EN20) | one FREE_CLIM land state in all 20 members | consistent (reference) |
| E10 | CLIMO_LND | the 20 DART analyses of E11 | one FREE_CLIM land state in all 20 members | mixed |
| E01 | CLIMO_ATM | FREE_CLIM + 20 DART perturbations (`FREECLIM_DARTPERT20`) | the 20 DART land analyses of E11 | mixed |
| E11 | DA_COUPLED | 20 DART analyses | 20 DART land analyses | consistent (coupled DA) |

The CLIM land is one FREE_CLIM state shared by every member, in CLIM_BASE and CLIMO_LND alike (only the atmosphere is perturbed: `ic_atm_freeclim_dartpert20`, `ic_climolnd_dartpert20/<date>/climo_ic_metadata.json`), so E00 and E10 start without land spread, while E01 and E11 carry the 20 DART land analyses. The land benefit therefore compares the DART land ensemble with a single climate land state (its information and its spread together, as intended for this question); the interaction is unaffected to first order, because E00 and E10 share the same land.

All perturbations and analyses come from the same 20 of the 40 DART members in a season (a different 20 in DJF and JJA). The DA runs keep the DART member numbers as labels; the members of each ensemble are exchangeable and are not paired across experiments, so every ensemble is used whole.

Benefits (contrasts of skill $S$, `contrasts`), centred on the full (coupled) DART initialization:

* atmosphere: $\Delta S_A = S_{11} - S_{01}$, the loss when the DART atmosphere is replaced by the reference atmosphere (with the DART land);
* land: $\Delta S_L = S_{11} - S_{10}$, the loss when the DART land is replaced by the DATM-forced land (with the DART atmosphere);
* interaction, the **land-atmosphere consistency** term: $I = S_{11} - S_{10} - S_{01} + S_{00}$, positive when the two consistent initializations (E11, E00) together score better than the two mixed ones (E10, E01), i.e. when matching land and atmosphere states adds skill beyond the two components separately;
* total: $S_{11} - S_{00} = \Delta S_A + \Delta S_L - I$.

In `dart_vs_capt` these are **benefits of DART relative to the CAPT-like reference**, not total atmospheric and land contributions to predictability: the reference atmosphere (ERA5) and land (DATM-forced) already carry information about the target dates. In `lnd_vs_atm` the reference states are free-running climate states with no information about the target dates beyond the prescribed SST, so the contrasts are closer to the total contributions of the DART atmosphere and land (as Richter et al. 2024, with free-running states in place of climatology).

Every initial state is complete (the ERA5 states are completed with CLIM_BASE fields; the superseded HYB_BASE, which lacked the radiative tracers, aerosols and chemistry, is in neither design: `METHODS.md`, initial-state QC). The sequential decomposition E00 -> E10 -> E11 (atmosphere $S_{10} - S_{00}$, land $S_{11} - S_{10}$) is available through `CASE_OVERRIDES` (see `InitBenefitSettings`).

## Inputs

Model: daily-mean TREFHT (mean of the four 6-hour means, `ts/6hourly`) of all 20 members of each experiment, lead days 1-56, on the 1-degree grid. Observations: ERA5 daily means (`ERA5/1x1/daily`). Climatologies over `clim_years`: observed (monthly means of the ERA5 daily series) and model (the free-running E3SMv3 AMIP runs), both interpolated to the verification days, as in notebook 02 (`util/fcst_init_data.py`).

## Method

1. Daily values of every member on the region's box (cached), and from them two time scales: weekly means (weeks 1-8 = lead days 1-7, ..., 50-56) and centred running means over `smooth_days` lead days at every lead day (the curves of the main figure, as in Richter et al. 2024).
2. Anomalies. Observed: ERA5 minus the observed climatology. Forecast: each member minus the observed (`"none"`) or the AMIP (`"amip"`) climatology (`biases`, as in notebook 02).
3. Skill: pattern ACC (centred, cos-lat weighted) over `region` of the ensemble mean, for each start and week (or lead day), averaged over the starts (`util/init_benefit.py`). With one year and 4 starts per season, 5 days apart, the gridpoint ACC across starts is too noisy to lead with; it is computed and saved as a secondary diagnostic (maps, weeks 3-4).
4. Uncertainty: a paired bootstrap (`n_boot`) resampling the starts within each season and the member indices jointly, with the same draw for every experiment; 2.5-97.5 % intervals.
5. Panel (c): each member k of E11 in turn is the truth; member index k is dropped from every ensemble (in E10 it shares the atmospheric initial state with the truth, in E01 the land initial state); forecasts and truth both use the AMIP climatology; skill is averaged over k. This is model-based potential predictability conditional on the prescribed SST, not an upper bound, and it can be inflated by an under-dispersive ensemble.

The main figure follows Richter et al. (2024, Fig. 1): curves at every lead day with the area down to zero shaded and the weeks-2-4 window (`window_weeks`) in grey; the weekly figure carries the bootstrap intervals. The vertical axis is ΔACC: like Richter's contributions (skill differences against runs with one component set to climatology), these are skill differences, here against the reference initialization. Panel (a) is a schematic of the hypothesis (atmospheric benefit early, land benefit relatively more important later); it is drawn, not fitted, and the results decide whether a crossover occurs.

## Limitations

One year, 8 starts (4 per season) that overlap in their verification periods: the intervals are wide and the starts are not independent samples. The benefits are conditional on the reference states (see Design). In `dart_vs_capt`, ERA5 verifies the forecasts and is also the E00 and E01 atmosphere, which favours them in week 1. See `METHODS.md` for the data and sampling choices common to the `hind_lead_eval` notebooks.

**The 05a-05c series** (DART initialization benefits for subseasonal 2-m temperature; each notebook reads the data it needs, computes, saves, then plots): `05a` the three-panel figure on daily running means; `05b` weekly benefits with bootstrap intervals and absolute skill; `05c` gridpoint-ACC benefit maps (DJF, JJA and both seasons in one figure). Settings: `InitBenefitSettings` in `configs/hind_lead_eval_cases.py`, design (`CONFIG_GROUP`): `dart_vs_capt` (question c) or `lnd_vs_atm` (question b); products: `util/init_benefit_workflow.py`; methods: `util/init_benefit.py`.

Every product is saved with the settings it was computed from and read back on the next run when those settings are unchanged; `FORCE_COMPUTE = True` recomputes this notebook's results, `FORCE_INPUTS = True` also re-reads the model output and observations. Model output is read only when a product has to be (re)computed.

## Outputs

`data/hind_lead_eval/init_benefit/`: `daily_<CONFIG>_<SEASON>_<VAR>.nc`, `daily_ref_<SEASON>_<VAR>.nc` (inputs shared by the designs), `<DESIGN>/init_benefit_curves_daily.csv`; figures `figure/hind_lead_eval/init_benefit/<DESIGN>/init_benefit_<BIAS>_<SET>.png`.

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options; plain values only, so `papermill -p` overrides work. Case definitions (experiments, regions, windows, options) are the settings defaults; change them for this notebook with `CASE_OVERRIDES`.

In [ ]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction, ELM layer thicknesses
AMIP_ROOT = OBSERVATION_ROOT / "E3SMv3"                        # free-running E3SMv3 DECK AMIP runs
WORKFLOW_NAME = "hind_lead_eval"

# Initialization configurations: every experiment (label, initial states, members, case name) is in
# configs/fcst_init_experiment_config.py (EXPERIMENTS). CONFIG_GROUP picks the set analysed, by question
# (CONFIG_GROUPS): "tracer_init" (a, incomplete vs completed ERA5 state), "lnd_vs_atm" (b, {CLIM, DART}
# atm x lnd), "dart_vs_capt" (c, {ERA5, DART} atm x {DATM, DART} lnd + CLIM), "coupling_consistency"
# (d, consistent vs mixed atm/lnd states), "all"; or a list of configuration keys.
# CASE_NAMES_OVERRIDE maps a selected key to another raw case name (INPUT_DATA_ROOT/<name>_<YYYYMMDD>00/<ENxx>).
CONFIG_GROUP = "dart_vs_capt"
CASE_NAMES_OVERRIDE = {}
# 05a-05c: CONFIG_GROUP is also the 2 x 2 design (INIT_BENEFIT_DESIGNS in configs/hind_lead_eval_cases.py):
# "dart_vs_capt" (c, reference E00 = ERA5CLIMFILL) or "lnd_vs_atm" (b, reference E00 = CLIM_BASE); each
# design saves its scores, maps and figures in its own subdirectory (<design>/)

CASE_OVERRIDES = {}     # changes to the InitBenefitSettings defaults (configs/hind_lead_eval_cases.py), e.g. {"max_members": 3}
FORCE_COMPUTE = False    # recompute this notebook's results even when they are saved with the same settings
FORCE_INPUTS = False     # also re-read the model output and observations (slow)


## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import STYLE_MAP, set_case_names
from configs.hind_lead_eval_cases import InitBenefitSettings, RunPaths
from util.hind_case_io import figure_saver, write_dataset, write_table
from util.init_benefit_workflow import InitBenefitCase

paths = RunPaths(*(Path(p) for p in (INPUT_DATA_ROOT, WORK_DIR, DIAGNOSTIC_OUTPUT_ROOT, OBSERVATION_ROOT,
                                     LAND_SUPPORT_DIR, AMIP_ROOT)), workflow_name=WORKFLOW_NAME)
from configs.fcst_init_experiment_config import select_configs
CONFIGS = select_configs(CONFIG_GROUP, CASE_NAMES_OVERRIDE)   # {configuration key: raw case name}
set_case_names(CONFIGS)   # the readers find each configuration's cases by these names
for c, name in CONFIGS.items():
    print(f"[CASE] {c}: {name}_<YYYYMMDD>00")
case = InitBenefitCase(InitBenefitSettings.for_design(CONFIG_GROUP, paths, **CASE_OVERRIDES), force_compute=FORCE_COMPUTE,
                   force_inputs=FORCE_INPUTS)
cfg = case.cfg


[CASE] CLIM_BASE: F20TR_ne30pg2_r05_IcoswISC30E3r5_CLIM_BASE_<YYYYMMDD>00
[CASE] ERA5CLIMFILL: F20TR_ne30pg2_r05_IcoswISC30E3r5_ERA5CLIMFILL_<YYYYMMDD>00
[CASE] DA_ATM: F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_ATM_<YYYYMMDD>00
[CASE] DA_LND_CLIMFILL: F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_LND_CLIMFILL_<YYYYMMDD>00
[CASE] DA_COUPLED: F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_COUPLED_<YYYYMMDD>00
[DESIGN] dart_vs_capt: benefits relative to CAPT-like (ERA5 atmosphere + DATM-forced land)
[DATA] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/init_benefit (daily caches)
[DATA] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/init_benefit/dart_vs_capt (scores, maps)
[FIGURE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/figure/hind_lead_eval/init_benefit/dart_vs_capt
[MEMBERS] DJF E00: 20: EN01, EN02, EN03, EN04, EN05, EN06, EN07, EN08, EN09, EN10, EN11, EN12, EN13, EN14, EN15, EN16, EN17, EN18, EN19, EN20
[MEMBERS] DJF E10: 20: 

## 4. Read input

The roles and the seasons in which every role's members are post-processed. The daily member values, observations and climatologies on the region's box (`daily_*.nc`, `daily_ref_*.nc`) are read when the scores must be computed.

In [ ]:
display(case.role_table())
print(f"[SEASONS] {case.seasons}; season sets: {case.sets}")

,role,config,atmosphere,land
0,E00,ERA5CLIMFILL,ERA5 (CLIM-filled) + DART pert.,DATM-forced
1,E10,DA_ATM,DART,DATM-forced
2,E01,DA_LND_CLIMFILL,ERA5 (CLIM-filled) + DART pert.,DART
3,E11,DA_COUPLED,DART,DART


[SEASONS] from the daily caches: ['DJF', 'JJA']
[SEASONS] ['DJF', 'JJA']; season sets: {'DJF': ['DJF'], 'JJA': ['JJA'], 'ALL': ['DJF', 'JJA']}


## 5. Process and save

Skill and contrast curves on `smooth_days` centred running means at every lead day: panel (b) per bias treatment and season set, panel (c) (held-out members, AMIP climatology), with paired-bootstrap intervals (`init_benefit_curves_daily.csv`). Shown: the contrasts at the end of each week, all seasons.

In [ ]:
table = case.curves("daily")
c = table[(table.kind == "contrast") & (table.set == "ALL") & (table.step % 7 == 0)]
display(c.pivot_table(index=["panel", "bias", "name"], columns="step", values="value", sort=False).round(3))

[CLIM] observed: ERA5 1x1 daily: monthly means of the daily series
[CLIM] model: E3SMv3 AMIP monthly (v3.LR.amip_0101, v3.LR.amip_0151; cmip_ts (time bounds))
[DAILY] E00 ERA5CLIMFILL DJF: {'init': 4, 'member': 20, 'lead': 56, 'lat': 30, 'lon': 360}
[DAILY] E10 DA_ATM DJF: {'init': 4, 'member': 20, 'lead': 56, 'lat': 30, 'lon': 360}
[DAILY] E01 DA_LND_CLIMFILL DJF: {'init': 4, 'member': 20, 'lead': 56, 'lat': 30, 'lon': 360}
[DAILY] E11 DA_COUPLED DJF: {'init': 4, 'member': 20, 'lead': 56, 'lat': 30, 'lon': 360}
[DAILY] E00 ERA5CLIMFILL JJA: {'init': 4, 'member': 20, 'lead': 56, 'lat': 30, 'lon': 360}
[DAILY] E10 DA_ATM JJA: {'init': 4, 'member': 20, 'lead': 56, 'lat': 30, 'lon': 360}
[DAILY] E01 DA_LND_CLIMFILL JJA: {'init': 4, 'member': 20, 'lead': 56, 'lat': 30, 'lon': 360}
[DAILY] E11 DA_COUPLED JJA: {'init': 4, 'member': 20, 'lead': 56, 'lat': 30, 'lon': 360}


## 6. Plot and show

The three-panel figure in the style of Richter et al. (2024, Fig. 1): (a) the hypothesis (schematic), (b) ERA5 verification, (c) held-out-member verification; the weeks-2-4 window in grey.

In [ ]:
# ============================ Figure settings: edit here and rerun this cell ============================
SHOW = True          # False: save the figures without showing them
FIGURE_DPI = 200
FONT_SIZE = 18       # base font size (pt); every label of every figure scales with it
FONT_SCALES = dict(label=1.0, title=1.0, tick=0.9, legend=1.0, suptitle=1.15, cell=0.75)  # x FONT_SIZE

from util.fcst_init_plotting import set_font_size
set_font_size(FONT_SIZE, **FONT_SCALES)
save = figure_saver(case.figure_dir, dpi=FIGURE_DPI, show=SHOW)
# =========================================================================================================

from util.init_benefit import plot_paper_figure

names = [n for n in ("atmosphere", "land", "interaction") if n in cfg.contrasts]
r = cfg.region
for name, set_seasons in case.sets.items():
    span = "+".join(set_seasons)
    pc = case.panel(table, "model member", name)
    for b in cfg.biases:
        save(plot_paper_figure(case.x_weeks, case.panel(table, "ERA5", name, b), pc, names=names, window=cfg.window_weeks,
                               title=f"DART initialization benefits relative to {cfg.reference}\n{cfg.variable} over land {r['lat'][0]:g}-"
                                     f"{r['lat'][1]:g}N ({span}; {cfg.smooth_days}-day running means; anomalies vs {b} "
                                     f"climatology; {case.n_members} members)"),
             f"init_benefit_{b}_{name}.png")